# OpenBayes：DeepSeek-R1-0528-Qwen3-8B 二分类 LoRA 微调（Jupyter 版）

- 目标：根据输入文本，输出 **“是/否”**（气候/天气变化相关性判别）  
- 数据（你已上传）：  
  - `/openbayes/input/input1/微调文本语料/train.jsonl`  
  - `/openbayes/input/input1/微调文本语料/dev.jsonl`  
  - `/openbayes/input/input1/微调文本语料/test.jsonl`  
- 环境：PyTorch 2.8 / Python 3.10 / Ubuntu 22.04 / CUDA 12.8 / RTX 5090  
- 日期：2026-01-24

# 本代码是从openbayes平台上运行，通过DeepSeek-R1-0528-Qwen3-8B微调模型进行文本分类的训练，得出气候语境相关的分类模型，代码里面包含全部的过程，包括相关的路径，训练过程以及训练结果，跑分结果，得出微调之后的模型准确性显著提升。


## 1. 环境与数据检查


In [2]:
import os, json, subprocess, sys, pathlib

print('Python:', sys.version)
print('Working dir:', os.getcwd())

try:
    out = subprocess.check_output(['nvidia-smi','-L'], text=True)
    print('\nGPU:\n', out)
except Exception as e:
    print('nvidia-smi not available:', e)


Python: 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:45:41) [GCC 13.3.0]
Working dir: /output

GPU:
 GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-d942849b-c950-50d9-43f2-0b5ce2cdda72)



In [3]:
DATA_DIR = '/openbayes/input/input1/微调文本语料'
train_path = f'{DATA_DIR}/train.jsonl'
dev_path   = f'{DATA_DIR}/dev.jsonl'
test_path  = f'{DATA_DIR}/test.jsonl'

for p in [train_path, dev_path, test_path]:
    print(p, '=>', 'OK' if os.path.exists(p) else 'NOT FOUND')

def head_jsonl(path, n=2):
    print('\n=== HEAD:', path, '===\n')
    with open(path, 'r', encoding='utf-8') as f:
        for i in range(n):
            line = f.readline()
            if not line:
                break
            obj = json.loads(line)
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:1200], '\n')

head_jsonl(train_path, 2)


/openbayes/input/input1/微调文本语料/train.jsonl => OK
/openbayes/input/input1/微调文本语料/dev.jsonl => OK
/openbayes/input/input1/微调文本语料/test.jsonl => OK

=== HEAD: /openbayes/input/input1/微调文本语料/train.jsonl ===

{
  "messages": [
    {
      "role": "system",
      "content": "你是一个专业的气候天气变化内容识别系统。请严格判断文本是否与气候变化以及天气变化直接相关，只回答“是”或“否”。"
    },
    {
      "role": "user",
      "content": "文本：同时，他们还圆满完成了40个点水源水质治理任务，结束了基层官兵喝不上达标饮用水的历史\n\n上下文：部党委先后8次召开专题会议，逐项研究审议整治方案，颁布《基层整治工程管理实施细则》。目前，这个部所属30多个营部、连队、执勤点的整治工作已经完成，并通过了检查验收。同时，他们还圆满完成了40个点水源水质治理任务，结束了基层官兵喝不上达标饮用水的历史。总政某部广播中心 培育当代军人树立核心价值观 总政某部广播中心主要领导带队到北京、天津等地方广播电台考察，邀请资深专家开设专业讲座，拓展业务人员视野。他们对20多个节目进行调整改版，延长节目播出时间，增强可听性，并健全节目听评制度，将编播人员的工作质量与表彰奖励紧密挂钩"
    },
    {
      "role": "assistant",
      "content": "是"
    }
  ]
} 

{
  "messages": [
    {
      "role": "system",
      "content": "你是一个专业的气候天气变化内容识别系统。请严格判断文本是否与气候变化以及天气变化直接相关，只回答“是”或“否”。"
    },
    {
      "role": "user",
      "content": "文本：——2008年，丙烯腈吸收塔尾气处理系统试车成功，脱硫除尘设施、蒸汽管网改造等随即配套实施，既解决了

## 2. 安装 ms-swift（一次性）
如果你已经装过并确认 `swift` 可用，可以跳过。


In [3]:
%cd /openbayes/home
!ls -lh


/output
total 45K
-rw-r--r-- 1 root root 7.8K Jan 24 03:54 OpenBayes_DeepSeekR1_0528_Qwen3_8B_finetune.ipynb
-rw-r--r-- 1 root root 6.8K Jan 23 12:38 autotune_batch_70b.py
drwxr-xr-x 3 root root    1 Jan 23 09:40 huggingface
drwxr-xr-x 2 root root    0 Jan 23 10:28 offload_70b
-rw-r--r-- 1 root root  15K Jan 24 03:34 run_climate_filter_2017.py
drwxr-xr-x 2 root root    2 Jan 23 12:59 yearly_csv_outputs_climate_labeled
-rw-r--r-- 1 root root  618 Jan 24 03:50 微调模型.ipynb
-rw-r--r-- 1 root root  14K Jan 24 03:46 调用模型.ipynb


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import os
if not os.path.exists('/openbayes/home/ms-swift'):
    !git clone https://github.com/modelscope/ms-swift.git
%cd /openbayes/home/ms-swift

!pip install -e . --user
!pip install -U transformers accelerate peft datasets --user
!pip install -U liger-kernel --user

os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ.get('PATH','')
!which swift
!swift --help | head -n 20


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/output/ms-swift
Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple
Obtaining file:///output/ms-swift
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.1 MB/s eta 0:00:00a 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Attempting uninstall: ms-swift
    Found existing installation: ms-swift 4.0.0.dev0
    Uninstalling ms-swift-4.0.0.dev0:
      Successfully uninstalled ms-swift-4.0.0.dev0
  Running setup.py develop for ms-swift
Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: datasets
    Found existing installation: datasets 3.6.0
    Uninstalling datasets-3.6.0:
      Successfully uninstalled datasets-3.6.0
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no

In [5]:
import os, pathlib

CACHE_DIR = "/openbayes/input/input0/models_cache"
pathlib.Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

# 推荐只用 HF_HOME（transformers 未来会移除 TRANSFORMERS_CACHE）
os.environ["HF_HOME"] = CACHE_DIR
# 可选：如果你之前设过 TRANSFORMERS_CACHE，直接删掉
os.environ.pop("TRANSFORMERS_CACHE", None)

print("HF_HOME =", os.environ["HF_HOME"])


HF_HOME = /openbayes/input/input0/models_cache


In [14]:
!python -m swift.cli.sft --help | head -n 60


usage: sft.py [-h] [--tuner_backend TUNER_BACKEND]

options:
  -h, --help            show this help message and exit
  --tuner_backend TUNER_BACKEND


## 3. 模型缓存目录（推荐）
把权重缓存到数据盘，避免容器重启后丢失。


In [6]:
import os, pathlib
CACHE_DIR = '/openbayes/input/input0/models_cache'
pathlib.Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
os.environ['MODELSCOPE_CACHE'] = CACHE_DIR
print('HF_HOME =', os.environ['HF_HOME'])


HF_HOME = /openbayes/input/input0/models_cache


## 4. 训练：LoRA SFT（DeepSeek-R1-0528-Qwen3-8B）
第一次会下载模型，可能较久。


In [7]:
import pathlib
OUT_DIR = '/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b'
pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('OUT_DIR:', OUT_DIR)


OUT_DIR: /openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b


In [15]:
import os, pathlib

DATA_DIR = "/openbayes/input/input1/微调文本语料"
train_path = f"{DATA_DIR}/train.jsonl"
dev_path   = f"{DATA_DIR}/dev.jsonl"

OUT_DIR = "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b"
pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

cmd = f"""
python -m swift.cli.sft \
  --model deepseek-ai/DeepSeek-R1-0528-Qwen3-8B \
  --train_type lora \
  --dataset "{train_path}" \
  --val_dataset "{dev_path}" \
  --output_dir "{OUT_DIR}" \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --max_length 1024 \
  --learning_rate 1e-4 \
  --lora_rank 8 \
  --lora_alpha 32 \
  --target_modules all-linear \
  --torch_dtype bfloat16 \
  --logging_steps 10 \
  --eval_steps 100 \
  --save_steps 100 \
  --save_total_limit 2 \
  --loss_scale ignore_empty_think
""".strip()

print(cmd)
!bash -lc "$cmd"


python -m swift.cli.sft   --model deepseek-ai/DeepSeek-R1-0528-Qwen3-8B   --train_type lora   --dataset "/openbayes/input/input1/微调文本语料/train.jsonl"   --val_dataset "/openbayes/input/input1/微调文本语料/dev.jsonl"   --output_dir "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b"   --num_train_epochs 1   --per_device_train_batch_size 1   --gradient_accumulation_steps 16   --max_length 1024   --learning_rate 1e-4   --lora_rank 8   --lora_alpha 32   --target_modules all-linear   --torch_dtype bfloat16   --logging_steps 10   --eval_steps 100   --save_steps 100   --save_total_limit 2   --loss_scale ignore_empty_think
[INFO:swift] Successfully registered `/output/ms-swift/swift/dataset/data/dataset_info.json`.
[WARNING:swift] `train_type` is deprecated, please use `tuner_type` instead.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from ModelScope Hub, model_id: deepseek-ai/DeepSeek-R1-0528-Qwen3-8B
[INFO:modelscope] Got 8 file

In [16]:
import os, json, pathlib

# ===== 你的数据路径 =====
DATA_DIR = "/openbayes/input/input1/微调文本语料"
train_path = f"{DATA_DIR}/train.jsonl"
dev_path   = f"{DATA_DIR}/dev.jsonl"
test_path  = f"{DATA_DIR}/test.jsonl"

# ===== 你的模型缓存路径（之前下载到这里）=====
CACHE_DIR = "/openbayes/input/input0/models_cache"
os.environ["HF_HOME"] = CACHE_DIR
os.environ.pop("TRANSFORMERS_CACHE", None)  # 未来会弃用，建议移除

# ===== 你的微调输出路径（训练已生成）=====
RUN_DIR = "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/v0-20260124-040114"
BEST_CKPT = f"{RUN_DIR}/checkpoint-400"   # best_model_checkpoint
LAST_CKPT = f"{RUN_DIR}/checkpoint-438"   # last_model_checkpoint

print("HF_HOME =", os.environ["HF_HOME"])
print("BEST_CKPT =", BEST_CKPT)
print("Files in BEST_CKPT:", os.listdir(BEST_CKPT)[:10])


HF_HOME = /openbayes/input/input0/models_cache
BEST_CKPT = /openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/v0-20260124-040114/checkpoint-400
Files in BEST_CKPT: ['trainer_state.json', 'README.md', 'training_args.bin', 'optimizer.pt', 'adapter_config.json', 'args.json', 'rng_state.pth', 'scheduler.pt', 'additional_config.json', 'adapter_model.safetensors']


In [17]:
def read_jsonl(path, n=1):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                break
            rows.append(json.loads(line))
    return rows

sample = read_jsonl(test_path, 1)[0]
print(json.dumps(sample, ensure_ascii=False, indent=2)[:1200])


{
  "messages": [
    {
      "role": "system",
      "content": "你是一个专业的气候天气变化内容识别系统。请严格判断文本是否与气候变化以及天气变化直接相关，只回答“是”或“否”。"
    },
    {
      "role": "user",
      "content": "文本：中国温室气体排放地区差异较大 如果不对奢侈、浪费型排放加以遏制，低碳社会就建立不起来 记者：中国已经提出了到2020年减少温室气体排放的行动目标，“十二五”中二氧化碳排放强度将成为约束性指标\n\n上下文：到中国的农村、中国的西部欠发达地区看一看，外国人就可以理解，中国仍然还是发展中国家。中国正在向发达国家迈进，但是客观地讲，我们和发达国家的差距还很大。中国温室气体排放地区差异较大 如果不对奢侈、浪费型排放加以遏制，低碳社会就建立不起来 记者：中国已经提出了到2020年减少温室气体排放的行动目标，“十二五”中二氧化碳排放强度将成为约束性指标。在这个指标的分解落实方面，您有哪些建议。潘家华：中国经济发展的地域差别很大，浙江、广东、上海、江苏等地的人均GDP现在已经达到中等发达国家水平了"
    },
    {
      "role": "assistant",
      "content": "是"
    }
  ]
}


In [19]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================
# 0. 强制离线（关键）
# =========================
os.environ["HF_HOME"] = "/openbayes/input/input0/models_cache"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# =========================
# 1. 本地模型路径（重点修改）
# =========================
BASE_MODEL_DIR = (
    "/openbayes/input/input0/models_cache/models/"
    "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"
)

BEST_CKPT = (
    "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/"
    "v0-20260124-040114/checkpoint-400"
)

# =========================
# 2. 加载 tokenizer（本地）
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
)

# =========================
# 3. 加载 base model（本地）
# =========================
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
    local_files_only=True,
)

# =========================
# 4. 加载 LoRA adapter
# =========================
model = PeftModel.from_pretrained(
    base_model,
    BEST_CKPT,
    device_map="cuda",
)

model.eval()

# =========================
# 5. 单条推理函数
# =========================
def generate_one(messages, max_new_tokens=8):
    """
    messages: [
      {"role":"system","content":...},
      {"role":"user","content":...}
    ]
    """
    # 用 Qwen / DeepSeek 的 chat template 拼 prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # 评测时关采样，保证稳定
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    # 只取模型最后的回答部分
    return text.strip().split(messages[-1]["content"])[-1].strip()

# =========================
# 6. 简单人工测试
# =========================
def read_jsonl(path, n=3):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rows.append(eval(line))
    return rows

test_path = "/openbayes/input/input1/微调文本语料/test.jsonl"

rows = read_jsonl(test_path, 3)
for i, r in enumerate(rows):
    msgs = r["messages"][:2]  # system + user
    print(f"\n--- Sample {i} ---")
    print("USER:", msgs[1]["content"][:120].replace("\n", " "), "...")
    print("MODEL:", generate_one(msgs))


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



--- Sample 0 ---
USER: 文本：中国温室气体排放地区差异较大 如果不对奢侈、浪费型排放加以遏制，低碳社会就建立不起来 记者：中国已经提出了到2020年减少温室气体排放的行动目标，“十二五”中二氧化碳排放强度将成为约束性指标  上下文：到中国的农村、中国的西部欠发达地 ...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


MODEL: <｜Assistant｜>是

--- Sample 1 ---
USER: 文本：浙江东阳推行低碳学车 为推动节能减排，今年浙江省东阳市将全市各驾校的模拟器统一集中，配备专业技术人员负责管理、教学和实现IC卡计时管理，对所有报名学车的学员实施模拟训练，再上路练习驾驶技能，有效节省了燃油和减少了尾气排放  上下文：浙 ...
MODEL: <｜Assistant｜>是

--- Sample 2 ---
USER: 文本：经改造之后，总部大楼的能源消耗将至少降低50％,温室气体排放量也将减少45％以上  上下文：改造工程已经于2008年5月启动，预定2013年竣工。据哈克介绍，总部大楼原有单层玻璃幕墙将被最新的双层中空玻璃板所替换，按计划这一工作将于2 ...
MODEL: <｜Assistant｜>是


In [21]:
import re
from tqdm import tqdm

def extract_yes_no(text: str):
    # 找到第一个 “是” 或 “否”
    m = re.search(r"(是|否)", text)
    return m.group(1) if m else None

def get_label_from_row(row):
    # 假设最后一个 messages 是 assistant 标注
    for msg in reversed(row["messages"]):
        if msg["role"] == "assistant":
            return extract_yes_no(msg["content"])
    return None

def predict_row(row):
    msgs = row["messages"][:2]  # system + user
    out = generate_one(msgs, max_new_tokens=8)
    return extract_yes_no(out)

# 跑评测
correct = 0
total = 0
missing = 0

with open(test_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

for row in tqdm(rows):
    y = get_label_from_row(row)
    p = predict_row(row)
    if y is None or p is None:
        missing += 1
        continue
    total += 1
    correct += int(p == y)

acc = correct / total if total else 0.0
print(f"Total evaluated: {total}, Correct: {correct}, Accuracy: {acc:.4f}, Missing: {missing}")


100%|██████████| 880/880 [01:59<00:00,  7.36it/s]

Total evaluated: 879, Correct: 781, Accuracy: 0.8885, Missing: 1


In [26]:
import os, json, pathlib, time
import torch

# ========= 1) 固定路径（全程用本地路径，不用模型ID） =========
MODEL_DIR = "/openbayes/input/input0/models_cache/models/deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"

RUN_DIR   = "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/v0-20260124-040114"
BEST_CKPT = f"{RUN_DIR}/checkpoint-400"   # best_model_checkpoint
LAST_CKPT = f"{RUN_DIR}/checkpoint-438"   # last_model_checkpoint（可选）

DATA_DIR  = "/openbayes/input/input1/微调文本语料"
TRAIN_PATH = f"{DATA_DIR}/train.jsonl"
DEV_PATH   = f"{DATA_DIR}/dev.jsonl"
TEST_PATH  = f"{DATA_DIR}/test.jsonl"

# 你后续两百万文本的输入文件（你自己改成真实路径）
BIG_INPUT_JSONL  = "/openbayes/input/input1/big_texts.jsonl"
BIG_OUTPUT_JSONL = "/openbayes/input/input1/big_texts_pred.jsonl"

# ========= 2) 强制离线，避免 transformers/hf-hub 再次下载 =========
CACHE_DIR = "/openbayes/input/input0/models_cache"
os.environ["HF_HOME"] = CACHE_DIR
os.environ.pop("TRANSFORMERS_CACHE", None)  # 避免未来弃用警告
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# 核心：离线
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

print("MODEL_DIR:", MODEL_DIR)
print("BEST_CKPT:", BEST_CKPT)
print("Offline flags set:",
      os.environ["TRANSFORMERS_OFFLINE"], os.environ["HF_HUB_OFFLINE"])

assert os.path.isdir(MODEL_DIR), "MODEL_DIR 不存在：请确认本地模型目录"
assert os.path.isdir(BEST_CKPT), "BEST_CKPT 不存在：请确认 LoRA ckpt 目录"
assert os.path.exists(TEST_PATH), "TEST_PATH 不存在：请确认 test.jsonl 路径"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


MODEL_DIR: /openbayes/input/input0/models_cache/models/deepseek-ai/DeepSeek-R1-0528-Qwen3-8B
BEST_CKPT: /openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/v0-20260124-040114/checkpoint-400
Offline flags set: 1 1
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


In [11]:
import gc, torch

# 如果你之前创建过这些变量（训练/推理用），删掉
for name in ["model", "base_model", "lora_model", "base", "trainer", "pipe"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Freed CUDA cache.")


Freed CUDA cache.


In [31]:
import re
from typing import List, Dict, Tuple

def read_jsonl(path: str, limit: int = None):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def ensure_chat_prompt(tokenizer, messages: List[Dict]) -> str:
    """
    优先用 tokenizer.apply_chat_template（Qwen 系一般可用）
    否则退化为简单拼接（保证能跑）
    """
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            pass

    # fallback：简单拼接
    sys_txt = ""
    user_txt = ""
    for m in messages:
        if m["role"] == "system":
            sys_txt += m["content"].strip() + "\n"
        elif m["role"] == "user":
            user_txt += m["content"].strip() + "\n"
    prompt = f"[SYSTEM]\n{sys_txt}\n[USER]\n{user_txt}\n[ASSISTANT]\n"
    return prompt

def parse_yes_no(text: str) -> Tuple[str, bool]:
    """
    从模型输出中解析最后一个 '是' 或 '否'。
    返回: (pred, ok)
    ok=False 表示无法解析（missing）
    """
    # 去掉空白
    t = text.strip()

    # 常见：输出里含很多内容，只要抓到最后一个“是/否”即可
    hits = re.findall(r"[是否]", t)
    if not hits:
        return ("", False)
    return (hits[-1], True)

def compute_metrics(y_true: List[str], y_pred: List[str]):
    """
    二分类，正类定义为 '是'（你任务里“相关”通常是正类）
    """
    assert len(y_true) == len(y_pred)

    tp = fp = tn = fn = 0
    for yt, yp in zip(y_true, y_pred):
        if yt == "是" and yp == "是":
            tp += 1
        elif yt == "否" and yp == "是":
            fp += 1
        elif yt == "否" and yp == "否":
            tn += 1
        elif yt == "是" and yp == "否":
            fn += 1

    total = tp + fp + tn + fn
    acc = (tp + tn) / total if total else 0.0

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0

    return {
        "total": total,
        "acc": acc,
        "precision_yes": prec,
        "recall_yes": rec,
        "f1_yes": f1,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn
    }


In [32]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_base_model(model_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=True,
        local_files_only=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.bfloat16,
        device_map="cuda" if torch.cuda.is_available() else "cpu",
        trust_remote_code=True,
        local_files_only=True
    )
    model.eval()
    return tokenizer, model

@torch.inference_mode()
def predict_batch(tokenizer, model, batch_messages: List[List[Dict]], max_new_tokens=8):
    prompts = [ensure_chat_prompt(tokenizer, msgs) for msgs in batch_messages]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id
    )

    texts = tokenizer.batch_decode(out, skip_special_tokens=True)
    preds = []
    oks = []
    for t in texts:
        p, ok = parse_yes_no(t)
        preds.append(p)
        oks.append(ok)
    return preds, oks, texts

def eval_on_test(tokenizer, model, test_path: str, batch_size=8, max_new_tokens=8,
                 save_errors_to: str = None):
    rows = read_jsonl(test_path)
    y_true, y_pred = [], []
    missing = 0
    errors = []

    # test.jsonl 你的格式：messages = [system, user, assistant]
    all_inputs = []
    all_labels = []
    for r in rows:
        msgs = r["messages"]
        all_inputs.append(msgs[:2])         # system + user
        all_labels.append(msgs[2]["content"].strip())  # gold = "是"/"否"

    t0 = time.time()
    for i in range(0, len(all_inputs), batch_size):
        batch_msgs = all_inputs[i:i+batch_size]
        batch_gold = all_labels[i:i+batch_size]

        preds, oks, raw_texts = predict_batch(tokenizer, model, batch_msgs, max_new_tokens=max_new_tokens)

        for j in range(len(preds)):
            gold = batch_gold[j]
            ok = oks[j]
            pred = preds[j] if ok else ""
            if not ok:
                missing += 1
            else:
                y_true.append(gold)
                y_pred.append(pred)

            # 记录错误/缺失样本，方便论文写错误分析
            if (not ok) or (pred != gold):
                errors.append({
                    "idx": i + j,
                    "gold": gold,
                    "pred": pred,
                    "ok": ok,
                    "user": batch_msgs[j][1]["content"][:500],
                    "raw_tail": raw_texts[j][-500:]
                })

    metrics = compute_metrics(y_true, y_pred)
    metrics["missing"] = missing
    metrics["seconds"] = round(time.time() - t0, 2)

    if save_errors_to:
        pathlib.Path(os.path.dirname(save_errors_to)).mkdir(parents=True, exist_ok=True)
        with open(save_errors_to, "w", encoding="utf-8") as f:
            for e in errors:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")

    return metrics

# ======= 跑 Base 评测 =======
base_tok, base_model = load_base_model(MODEL_DIR)

base_metrics = eval_on_test(
    base_tok, base_model, TEST_PATH,
    batch_size=8,
    max_new_tokens=8,
    save_errors_to=f"{RUN_DIR}/base_errors.jsonl"
)

print("BASE metrics:", base_metrics)


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

BASE metrics: {'total': 880, 'acc': 0.46704545454545454, 'precision_yes': 0.5756929637526652, 'recall_yes': 0.5, 'f1_yes': 0.5351833498513379, 'tp': 270, 'fp': 199, 'tn': 141, 'fn': 270, 'missing': 0, 'seconds': 65.94}


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ===== 必须定义的路径 =====
MODEL_DIR = "/openbayes/input/input0/models_cache/models/deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"
BEST_CKPT = "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/v0-20260124-040114/checkpoint-400"
TEST_PATH = "/openbayes/input/input1/微调文本语料/test.jsonl"
RUN_DIR = "/openbayes/input/input1/eval_runs"


In [17]:
import os, json, re, time, gc, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================
# 0) 路径：按你现有的来
# =========================
BASE_MODEL_DIR = "/openbayes/input/input0/models_cache/models/deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"
LORA_CKPT      = "/openbayes/input/input1/finetune_outputs_r1_0528_qwen3_8b/v0-20260124-040114/checkpoint-400"  # best checkpoint
TEST_PATH      = "/openbayes/input/input1/微调文本语料/test.jsonl"
RUN_DIR        = "/openbayes/input/input1/eval_runs"
pathlib.Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

YES, NO = "是", "否"

# =========================
# 1) 工具函数
# =========================
def parse_yes_no(text: str):
    """从生成文本里提取 是/否；提取不到返回 None"""
    if text is None:
        return None
    # 找最后一次出现的“是/否”
    m = re.findall(r"[是否]", text)
    if m:
        return m[-1]
    t = text.strip().lower()
    if re.search(r"\byes\b|true", t):
        return YES
    if re.search(r"\bno\b|false", t):
        return NO
    return None

@torch.no_grad()
def generate_one(tokenizer, model, messages, max_new_tokens=64):
    """
    messages: [{"role":"system","content":...},{"role":"user","content":...}]
    返回：生成出来的新增文本（不包含prompt部分）
    """
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")

    # ✅ 强制 inputs 上 GPU（因为我们后面也强制 model 上 GPU）
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    in_len = inputs["input_ids"].shape[-1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,   # ✅ 给够，避免只输出 <think> 截断
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    gen_ids = out[0][in_len:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return gen_text

def eval_on_test(tokenizer, model, test_path, max_new_tokens=64, save_errors_to=None, limit=None):
    """
    输出：acc/precision/recall/f1 + confusion + missing + seconds
    同时把错误样本（含 gen_text）写到 jsonl，方便你论文分析
    """
    t0 = time.time()
    tp=fp=tn=fn=0
    missing=0
    total=0

    err_f = open(save_errors_to, "w", encoding="utf-8") if save_errors_to else None

    with open(test_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            ex = json.loads(line)
            total += 1
            if limit and total > limit:
                break

            msgs = ex["messages"][:2]
            gold = ex["messages"][2]["content"].strip()
            gold = YES if gold.startswith(YES) else NO

            gen_text = generate_one(tokenizer, model, msgs, max_new_tokens=max_new_tokens)
            pred = parse_yes_no(gen_text)

            if pred is None:
                missing += 1
                if err_f:
                    err_f.write(json.dumps({
                        "gold": gold,
                        "pred": None,
                        "gen_text": gen_text,
                        "user_head": msgs[1]["content"][:200]
                    }, ensure_ascii=False) + "\n")
                continue

            pred = YES if pred == YES else NO

            if pred==YES and gold==YES: tp += 1
            elif pred==YES and gold==NO: fp += 1
            elif pred==NO  and gold==NO: tn += 1
            elif pred==NO  and gold==YES: fn += 1

            if err_f and pred != gold:
                err_f.write(json.dumps({
                    "gold": gold,
                    "pred": pred,
                    "gen_text": gen_text,
                    "user_head": msgs[1]["content"][:200]
                }, ensure_ascii=False) + "\n")

    if err_f:
        err_f.close()

    denom = tp+fp+tn+fn
    acc = (tp+tn)/denom if denom else 0.0
    precision_yes = tp/(tp+fp) if (tp+fp) else 0.0
    recall_yes = tp/(tp+fn) if (tp+fn) else 0.0
    f1_yes = 2*precision_yes*recall_yes/(precision_yes+recall_yes) if (precision_yes+recall_yes) else 0.0

    return {
        "total": total,
        "acc": acc,
        "precision_yes": precision_yes,
        "recall_yes": recall_yes,
        "f1_yes": f1_yes,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "missing": missing,
        "seconds": round(time.time()-t0, 2)
    }

# =========================
# 2) 清显存，加载模型（强制整模型上 GPU）
# =========================
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_DIR, trust_remote_code=True, local_files_only=True
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    local_files_only=True,
).to("cuda")     # ✅ 强制整个 base 上 GPU
base.eval()

lora_model = PeftModel.from_pretrained(
    base,
    LORA_CKPT,
    is_trainable=False
).to("cuda")     # ✅ LoRA 也在 GPU
lora_model.eval()

# =========================
# 3) 先打印 5 条样本，确认能生成“是/否”
# =========================
print("==== Sanity check 5 samples ====")
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for i in range(5):
        ex = json.loads(f.readline())
        msgs = ex["messages"][:2]
        gold = ex["messages"][2]["content"].strip()
        gen_text = generate_one(tokenizer, lora_model, msgs, max_new_tokens=64)
        pred = parse_yes_no(gen_text)
        print(f"\n--- sample {i} ---")
        print("gold:", gold)
        print("gen_text_head:", repr(gen_text[:120]))
        print("pred:", pred)

# =========================
# 4) 正式评测
# =========================
lora_metrics = eval_on_test(
    tokenizer, lora_model, TEST_PATH,
    max_new_tokens=64,
    save_errors_to=f"{RUN_DIR}/lora_errors_fixed.jsonl"
)

print("\nLORA metrics:", lora_metrics)
print("Errors saved to:", f"{RUN_DIR}/lora_errors_fixed.jsonl")


Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

==== Sanity check 5 samples ====

--- sample 0 ---
gold: 是
gen_text_head: '是'
pred: 是

--- sample 1 ---
gold: 是
gen_text_head: '是'
pred: 是

--- sample 2 ---
gold: 是
gen_text_head: '是'
pred: 是

--- sample 3 ---
gold: 否
gen_text_head: '否'
pred: 否

--- sample 4 ---
gold: 是
gen_text_head: '是'
pred: 是

LORA metrics: {'total': 880, 'acc': 0.8863636363636364, 'precision_yes': 0.9, 'recall_yes': 0.9166666666666666, 'f1_yes': 0.908256880733945, 'tp': 495, 'fp': 55, 'tn': 285, 'fn': 45, 'missing': 0, 'seconds': 120.82}
Errors saved to: /openbayes/input/input1/eval_runs/lora_errors_fixed.jsonl


In [18]:
import os, csv, json, random, pathlib

RUN_DIR = "/openbayes/input/input1/eval_runs"
pathlib.Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

# 你给过的 BASE metrics（直接写死，避免变量丢失）
base_metrics = {
  'total': 880, 'acc': 0.46704545454545454,
  'precision_yes': 0.5756929637526652, 'recall_yes': 0.5, 'f1_yes': 0.5351833498513379,
  'tp': 270, 'fp': 199, 'tn': 141, 'fn': 270, 'missing': 0, 'seconds': 65.94
}

# 你刚跑出来的 LORA metrics（直接写死，避免变量丢失）
lora_metrics = {
  'total': 880, 'acc': 0.8863636363636364,
  'precision_yes': 0.9, 'recall_yes': 0.9166666666666666, 'f1_yes': 0.908256880733945,
  'tp': 495, 'fp': 55, 'tn': 285, 'fn': 45, 'missing': 0, 'seconds': 120.82
}

# ===== 1) 导出 CSV（论文表格源文件）=====
report_csv = f"{RUN_DIR}/paper_metrics.csv"
with open(report_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["model", "total", "acc", "precision_yes", "recall_yes", "f1_yes", "missing", "tp", "fp", "tn", "fn", "seconds"])
    w.writerow(["BASE", base_metrics["total"], base_metrics["acc"], base_metrics["precision_yes"], base_metrics["recall_yes"], base_metrics["f1_yes"],
                base_metrics["missing"], base_metrics["tp"], base_metrics["fp"], base_metrics["tn"], base_metrics["fn"], base_metrics["seconds"]])
    w.writerow(["LORA(best_ckpt)", lora_metrics["total"], lora_metrics["acc"], lora_metrics["precision_yes"], lora_metrics["recall_yes"], lora_metrics["f1_yes"],
                lora_metrics["missing"], lora_metrics["tp"], lora_metrics["fp"], lora_metrics["tn"], lora_metrics["fn"], lora_metrics["seconds"]])

# ===== 2) 导出混淆矩阵文本 =====
cm_txt = f"{RUN_DIR}/confusion_matrix.txt"
with open(cm_txt, "w", encoding="utf-8") as f:
    f.write("Confusion Matrix (Positive='是')\n\n")
    f.write("BASE:\n")
    f.write(f"TP={base_metrics['tp']}  FP={base_metrics['fp']}\n")
    f.write(f"FN={base_metrics['fn']}  TN={base_metrics['tn']}\n\n")
    f.write("LORA:\n")
    f.write(f"TP={lora_metrics['tp']}  FP={lora_metrics['fp']}\n")
    f.write(f"FN={lora_metrics['fn']}  TN={lora_metrics['tn']}\n")

# ===== 3) 从 LoRA 错误样本里抽样（用于论文“错误分析”）=====
err_path = f"{RUN_DIR}/lora_errors_fixed.jsonl"
sample_out = f"{RUN_DIR}/lora_error_samples_50.jsonl"

errors = []
if os.path.exists(err_path):
    with open(err_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                errors.append(json.loads(line))

random.shuffle(errors)
errors = errors[:50]

with open(sample_out, "w", encoding="utf-8") as f:
    for e in errors:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

print("Saved CSV:", report_csv)
print("Saved confusion matrix:", cm_txt)
print("Saved error sample:", sample_out)
print("Total errors in file:", len(errors))


Saved CSV: /openbayes/input/input1/eval_runs/paper_metrics.csv
Saved confusion matrix: /openbayes/input/input1/eval_runs/confusion_matrix.txt
Saved error sample: /openbayes/input/input1/eval_runs/lora_error_samples_50.jsonl
Total errors in file: 50
